> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

# Olist 主线第1课：把九张表真正读懂

## 本课定位
这是你提出的学习顺序：来源 → 表结构 → 粒度 → 键 → 关联 → 时间 → 质量 → 数据理解总结，然后才定义业务目标、标签、预测时点和验证方案。
前面 notebook 保留作参考；从本文件继续主线，避免反复切换。你已学会的表结构概念不需要重学。

我们不默认复购、不默认批准订单、不默认六个月或90天。本课不删除原始记录，不选择模型。
九张表也不是必须一次合成一张宽表：先画清关系，再根据将来的分析单位决定怎样汇总。

## 0. 来源与可复现性
直接取得位置：你们GitHub仓库中分发的原始ZIP；发布方为Olist。仓库清单记录上游为Kaggle Brazilian E-Commerce Public Dataset by Olist，版本2，许可CC BY-NC-SA 4.0。
https://github.com/CercaTrovato/olist-repurchase-analysis
这些信息来自仓库来源清单，并非本课重新独立认证了数据完整性。我们会校验原包SHA-256以确认使用的是同一份文件。

代码里：`shape`读行列数；`head`看样例；`isna`找缺失；`nunique`数不同值；`duplicated`找重复。它们用于检查，不是自动清洗规则。


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd

DATA = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/olist-v2.zip').is_file()), Path.cwd()) / 'data' / 'olist-v2.zip'
assert DATA.exists(), '请在本notebook所在的olist_learning目录运行，旁边需要data/olist-v2.zip'
expected = '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == expected
with ZipFile(DATA) as z:
    files = sorted(n for n in z.namelist() if n.endswith('.csv'))
    frames = {}
    for name in files:
        # 邮编作为代码读取，避免把代码当作连续数量；不填补、不删记录。
        with z.open(name) as f:
            frames[name] = pd.read_csv(f, dtype={
                'customer_zip_code_prefix': 'string',
                'seller_zip_code_prefix': 'string',
                'geolocation_zip_code_prefix': 'string'
            })
print('ZIP校验通过；CSV数量：', len(files))
print(pd.DataFrame([{'文件': n, '行数': len(frames[n]), '列数': frames[n].shape[1]}
                    for n in files]).to_string(index=False))


## 1. 逐张表列出全部字段：含义、读取类型、缺失和不同值数
本数据九张表合计52个字段位置，同名字段在不同表分别计数。CSV读取类型不等于业务类型：时间可能暂时读成object；编号不是可以求平均的测量值。
以下字段解释用于理解结构。少数时间字段的业务生成机制仍应核查来源，不凭名字推定预测时可用。


### customers：olist_customers_dataset.csv

**一行是什么：**订单侧客户资料；customer_id在本文件中每行唯一，不等于跨订单买家身份。

| 字段 | 理解用途 |
|---|---|
| customer_id | 连接订单的客户编号；候选键 |
| customer_unique_id | 跨订单识别买家的编号；本表不唯一 |
| customer_zip_code_prefix | 邮编前缀；代码而非数量 |
| customer_city | 客户城市 |
| customer_state | 客户州代码 |

In [ ]:
t = frames['olist_customers_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### orders：olist_orders_dataset.csv

**一行是什么：**一笔订单。

| 字段 | 理解用途 |
|---|---|
| order_id | 订单编号；候选键 |
| customer_id | 连接customers的编号 |
| order_status | 记录的订单状态 |
| order_purchase_timestamp | 下单时间 |
| order_approved_at | 批准时间 |
| order_delivered_carrier_date | 交给承运方的时间记录 |
| order_delivered_customer_date | 送达客户的时间记录 |
| order_estimated_delivery_date | 预计送达日期 |

In [ ]:
t = frames['olist_orders_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### order_items：olist_order_items_dataset.csv

**一行是什么：**订单内一条商品明细；一行不一定代表一种不同商品。

| 字段 | 理解用途 |
|---|---|
| order_id | 所属订单编号；复合键组成部分 |
| order_item_id | 订单内明细序号；复合键组成部分 |
| product_id | 商品编号 |
| seller_id | 卖家编号 |
| shipping_limit_date | 明细对应的发货截止时间字段；使用前应核查来源语义 |
| price | 该条明细商品价格 |
| freight_value | 该条明细记录的运费 |

In [ ]:
t = frames['olist_order_items_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### payments：olist_order_payments_dataset.csv

**一行是什么：**订单的一条支付记录；一单可以多条。

| 字段 | 理解用途 |
|---|---|
| order_id | 所属订单编号；复合键组成部分 |
| payment_sequential | 订单内支付序号；复合键组成部分 |
| payment_type | 支付方式 |
| payment_installments | 分期期数 |
| payment_value | 该条支付金额 |

In [ ]:
t = frames['olist_order_payments_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### reviews：olist_order_reviews_dataset.csv

**一行是什么：**与某订单关联的一条评价记录；review_id单列不能在此CSV中唯一识别行。

| 字段 | 理解用途 |
|---|---|
| review_id | 评价编号；实测不唯一 |
| order_id | 关联订单编号 |
| review_score | 评价分数；有序评分 |
| review_comment_title | 评价标题；文本缺失不自动等于负面评价 |
| review_comment_message | 评价正文 |
| review_creation_date | 评价创建日期字段；使用前核查业务含义 |
| review_answer_timestamp | 评价提交/回答时间字段 |

In [ ]:
t = frames['olist_order_reviews_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### products：olist_products_dataset.csv

**一行是什么：**一个商品的属性。

| 字段 | 理解用途 |
|---|---|
| product_id | 商品编号；候选键 |
| product_category_name | 原始品类名称 |
| product_name_lenght | 商品名长度；保留原始拼写 |
| product_description_lenght | 商品描述长度；保留原始拼写 |
| product_photos_qty | 商品图片数量 |
| product_weight_g | 重量，克 |
| product_length_cm | 长度，厘米 |
| product_height_cm | 高度，厘米 |
| product_width_cm | 宽度，厘米 |

In [ ]:
t = frames['olist_products_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### sellers：olist_sellers_dataset.csv

**一行是什么：**一个卖家的资料。

| 字段 | 理解用途 |
|---|---|
| seller_id | 卖家编号；候选键 |
| seller_zip_code_prefix | 卖家邮编前缀 |
| seller_city | 卖家城市 |
| seller_state | 卖家州代码 |

In [ ]:
t = frames['olist_sellers_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### geolocation：olist_geolocation_dataset.csv

**一行是什么：**一条邮编前缀相关的坐标记录；既不是每个邮编一行，也未证明每行唯一。

| 字段 | 理解用途 |
|---|---|
| geolocation_zip_code_prefix | 邮编前缀；实测不唯一 |
| geolocation_lat | 纬度 |
| geolocation_lng | 经度 |
| geolocation_city | 记录的城市名 |
| geolocation_state | 州代码 |

In [ ]:
t = frames['olist_geolocation_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### translation：product_category_name_translation.csv

**一行是什么：**一个品类名称的英文对照。

| 字段 | 理解用途 |
|---|---|
| product_category_name | 原始品类名称；候选键 |
| product_category_name_english | 英文品类名称 |

In [ ]:
t = frames['product_category_name_translation.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


## 2. 主键不是看字段名猜出来的
主键：我们选定的、非空且唯一识别一行的键；可以由多列共同组成。
候选键：满足条件、可以被选作主键的字段组合。CSV没有数据库约束，本课先做“这份文件的候选键检查”，不声称已经获得完整业务模式。

检查顺序：有没有缺失 → 单列是否唯一 → 必要时检查复合键 → 结合一行的业务含义解释。
即使某列在一个样本里唯一，也不能单凭这一点认定它是业务上长期可靠的键。


In [ ]:
key_candidates = [
    ('orders', 'olist_orders_dataset.csv', ['order_id']),
    ('orders_customer_id', 'olist_orders_dataset.csv', ['customer_id']),
    ('customers', 'olist_customers_dataset.csv', ['customer_id']),
    ('customers_buyer_id', 'olist_customers_dataset.csv', ['customer_unique_id']),
    ('items_order_only', 'olist_order_items_dataset.csv', ['order_id']),
    ('items_composite', 'olist_order_items_dataset.csv', ['order_id','order_item_id']),
    ('payments_composite', 'olist_order_payments_dataset.csv', ['order_id','payment_sequential']),
    ('reviews_id_only', 'olist_order_reviews_dataset.csv', ['review_id']),
    ('reviews_composite', 'olist_order_reviews_dataset.csv', ['review_id','order_id']),
    ('products', 'olist_products_dataset.csv', ['product_id']),
    ('sellers', 'olist_sellers_dataset.csv', ['seller_id']),
    ('translation', 'product_category_name_translation.csv', ['product_category_name']),
    ('geolocation_prefix', 'olist_geolocation_dataset.csv', ['geolocation_zip_code_prefix'])
]
key_rows=[]
for label, filename, keys in key_candidates:
    t=frames[filename]
    nulls=int(t[keys].isna().any(axis=1).sum())
    duplicates=int(t.duplicated(keys).sum())
    key_rows.append({'检查': label, '字段组合': ' + '.join(keys), '空键行数': nulls,
                     '重复键行数_超出首行': duplicates, '本文件非空唯一': nulls==0 and duplicates==0})
print(pd.DataFrame(key_rows).to_string(index=False))


**结果解释：**
- orders.order_id、customers.customer_id、products.product_id、sellers.seller_id和品类对照表的原始品类名，都在这份文件中非空且唯一。
- 商品明细需要(order_id,order_item_id)；支付需要(order_id,payment_sequential)。序号只在同一订单内识别明细。
- review_id存在814个超出首次出现的重复；(review_id,order_id)组合无重复，可作为本文件中的候选复合键，但仍需了解评价生成机制。不能删掉所有重复review_id而不检查它关联的订单。
- geolocation的邮编前缀不是主键；全表还有261831个超出首次出现的整行重复。先记录，不在不了解用途时直接决定去重或取平均坐标。
- orders.customer_id在本文件也唯一，因此orders与customers按customer_id连接是观测到的1:1；不意味着一个自然买家只会下一个订单。跨订单买家由customer_unique_id关联。


## 3. 外键与基数：逻辑上能连，不等于随便merge
外键是子表中引用父表键的列。这里是CSV，所以检查的是逻辑引用关系，不是数据库已经强制执行的外键约束。
检验：父端是否唯一？子端有没有空键？非空键在父端能否找到？每个父记录可以对应几条子记录？

结构地图：

```text
隐含买家实体(customer_unique_id)
  └─ customers -- customer_id --> orders
        │                          ├─ order_items -- product_id --> products -- category --> translation
        │                          │       └─ seller_id --> sellers
        │                          ├─ payments
        │                          └─ reviews
        └─ customer_zip_code_prefix ─┐
                         sellers.zip ├─ geolocation（邮编不是唯一，不能当普通唯一维表）
```

这里的orders到items/payments/reviews是一对多（有些订单无子记录）。items到products/sellers是多对一。


In [ ]:
relations = [
    ('orders→customers','olist_orders_dataset.csv','customer_id','olist_customers_dataset.csv','customer_id'),
    ('items→orders','olist_order_items_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('payments→orders','olist_order_payments_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('reviews→orders','olist_order_reviews_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('items→products','olist_order_items_dataset.csv','product_id','olist_products_dataset.csv','product_id'),
    ('items→sellers','olist_order_items_dataset.csv','seller_id','olist_sellers_dataset.csv','seller_id'),
    ('products→translation','olist_products_dataset.csv','product_category_name','product_category_name_translation.csv','product_category_name'),
    ('customers→geo','olist_customers_dataset.csv','customer_zip_code_prefix','olist_geolocation_dataset.csv','geolocation_zip_code_prefix'),
    ('sellers→geo','olist_sellers_dataset.csv','seller_zip_code_prefix','olist_geolocation_dataset.csv','geolocation_zip_code_prefix')
]
relation_rows=[]
for label,cn,ck,pn,pk in relations:
    child,parent=frames[cn],frames[pn]
    orphan=child[ck].notna() & ~child[ck].isin(parent[pk].dropna())
    parent_values=parent[pk].dropna().drop_duplicates()
    relation_rows.append({'关系': label,'父端键唯一': parent[pk].is_unique,
        '子端空键行数': int(child[ck].isna().sum()),'非空但父端找不到_行数': int(orphan.sum()),
        '父端找不到_不同键数': int(child.loc[orphan,ck].nunique()),
        '无子记录的父键数': int((~parent_values.isin(child[ck].dropna())).sum())})
print(pd.DataFrame(relation_rows).to_string(index=False))


**怎样解释这些关联？**
- 明细、支付、评价的非空order_id都能在订单表找到；但反方向并非每笔订单都有子记录。
- 775笔订单没有商品明细，1笔没有支付记录，768笔没有评价。是否排除它们，必须等业务问题确定，不把无评价自动当作差评。
- 商品品类有610个空值；另有13个商品的非空品类在对照表找不到。缺原始品类与缺英文翻译不是同一种缺失。
- 278条客户资料的邮编前缀和7个卖家的邮编前缀找不到地理记录。邮编映射也不是100%覆盖。
- geo父端键不唯一，因此不能直接用many_to_one验证通过。需根据未来用途决定邮编层级的坐标或其他地理映射，不能默认每个前缀对应唯一城市、唯一坐标。


## 4. 真正做一次小范围连接：为什么九表直接拼接会错？
先选一笔同时有2条商品明细、2条支付的真实订单。在原始粒度按order_id直接连接，两边会两两组合成4行。
这里演示错误与核查，不对全数据做错误大连接。


In [ ]:
items = frames['olist_order_items_dataset.csv']
payments = frames['olist_order_payments_dataset.csv']
ic=items.groupby('order_id').size()
pc=payments.groupby('order_id').size()
sizes=pd.concat([ic.rename('item_rows'),pc.rename('payment_rows')],axis=1).fillna(0)
example_id=sizes.loc[sizes['item_rows'].eq(2)&sizes['payment_rows'].eq(2)].index[0]
one_items=items.loc[items['order_id'].eq(example_id)]
one_payments=payments.loc[payments['order_id'].eq(example_id)]
bad=one_items.merge(one_payments,on='order_id',validate='many_to_many')
print('订单编号：',example_id)
print('原始明细：')
print(one_items[['order_item_id','price','freight_value']].to_string(index=False))
print('原始支付：')
print(one_payments[['payment_sequential','payment_type','payment_value']].to_string(index=False))
print('直接拼接后的组合：')
print(bad[['order_item_id','payment_sequential','price','payment_value']].to_string(index=False))
print('拼接行数：',len(bad))
print('商品金额原始合计：',round(one_items['price'].sum(),2),
      '拼接后错误合计：',round(bad['price'].sum(),2))
print('支付原始合计：',round(one_payments['payment_value'].sum(),2),
      '拼接后错误合计：',round(bad['payment_value'].sum(),2))

# 如果目标是一行一订单，两边先各自汇总成一行，再连接。
item_order=one_items.groupby('order_id').agg(goods_amount=('price','sum'),freight=('freight_value','sum'))
payment_order=one_payments.groupby('order_id').agg(payment_amount=('payment_value','sum'))
good=item_order.join(payment_order,validate='one_to_one')
print('演示订单层级的正确汇总：')
print(good.round(2).to_string())
assert len(good)==1
assert abs(good['goods_amount'].iloc[0]-one_items['price'].sum())<1e-9


**结果解释：** 这笔订单商品金额298.90，支付375.73；商品金额加明细运费76.83恰好等于支付金额。这只是本例核对成功，不是对全部订单的结论。
直接拼接会把商品金额误算为597.80、支付误算为751.46。
`validate='many_to_many'`并不能防止这种膨胀，只是允许多对多。因此“merge没报错”不等于分析正确。
若未来分析粒度是订单，可先汇总明细、支付、评价后分别连订单；若分析商品明细，订单级金额不能在每条明细重复求和。
客户层级还需要先判断跨订单买家，再汇总；今天只理解，不提前决定最终客户资格。


## 5. 与数据库1NF、2NF、3NF有什么联系？
你提到的通常写作1NF、2NF、3NF（第一、第二、第三范式）。思路相关，但不需要把九张现成表重新机械拆一遍。

函数依赖A→B：在业务定义中，一个A值决定一个B值。数据检查可以发现反例，样本恰好没有反例不等于证明业务上永远成立。

### 1NF：按单值字段组织记录
没有把“一张订单的所有商品列表”塞进一个重复组列，而是用order_items逐条记录，便于关联与统计。长评论作为一个文本值本身不违反原子性。不要用“列中有逗号”判断违反1NF。

### 2NF：非主属性不依赖复合候选键的一部分
若以(order_id,order_item_id)作为明细键，却把order_purchase_timestamp反复存进明细表：订单时间只依赖order_id，属于部分依赖的典型例子。把订单属性放orders、明细属性放items有助于避免这类冗余。

### 3NF：去除不合适的传递依赖（正式定义：每个非平凡函数依赖X→A，X是超键或A是主属性）
例如把英文品类名存进每个商品：product_id→原始品类→英文品类，英文名称会随同一品类重复。独立的品类对照表可以避免这种冗余。
仅凭这些例子不能认证九张表全部满足3NF；还需完整候选键、业务函数依赖和来源说明。geo和reviews的键问题尤其需要先处理。

### 数据库存储与分析宽表不同
规范化服务于减少冗余和维护一致性；建模特征表常常有意汇总/反规范化成每个分析对象一行。不是先把九表全merge，就等于完成规范化或特征工程。


## 6. 全时期时间检查：按每种事件分别检查，不先筛订单
时间列先转换并报告解析失败，不因缺失立刻删行。下单、批准、送达、预计送达是不同事件；它们的最大日期不能互相替代。


In [ ]:
date_specs = {
    'olist_orders_dataset.csv': ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'],
    'olist_order_items_dataset.csv': ['shipping_limit_date'],
    'olist_order_reviews_dataset.csv': ['review_creation_date','review_answer_timestamp']
}
date_rows=[]
parsed_dates={}
for filename, cols in date_specs.items():
    for col in cols:
        original=frames[filename][col]
        parsed=pd.to_datetime(original,errors='coerce')
        parsed_dates[(filename,col)]=parsed
        date_rows.append({'表': filename,'字段': col,'最早': str(parsed.min()),'最晚': str(parsed.max()),
                          '原始缺失': int(original.isna().sum()),
                          '非空但解析失败': int((original.notna()&parsed.isna()).sum())})
print(pd.DataFrame(date_rows).to_string(index=False))
p=parsed_dates[('olist_orders_dataset.csv','order_purchase_timestamp')]
a=parsed_dates[('olist_orders_dataset.csv','order_approved_at')]
months=pd.period_range(p.min().to_period('M'),p.max().to_period('M'),freq='M')
monthly=pd.DataFrame({'原始下单数':p.groupby(p.dt.to_period('M')).size(),
                       '有批准时间的订单数_按批准月':a.groupby(a.dt.to_period('M')).size()}).reindex(months).fillna(0).astype(int)
print('全部月份：')
print(monthly.to_string())


**结果解释：** 下单记录覆盖2016-09至2018-10，但不同事件日期的边界不同。2016年末、2018年末的月度记录稀疏，是数据覆盖待核查事项；本课不据此直接选择训练或测试时期。
预计送达可以晚于最后下单日，因为它是预计日期，不代表那一天有真实新订单。
月度记录为0只说明本文件没观察到该事件，不足以证明现实中没有业务。
时间先后异常可作为核查线索，不应未经来源核实就批量删除。尤其预计日期常为零点，应先决定按日期还是精确时间比较。


## 7. 质量问题怎样记录，而不是自动清洗？
每项记录“发现、可能影响、待核查、拟处理方式及理由”。目前先标记待定。

| 发现 | 可能影响 | 现在做什么 |
|---|---|---|
| 主订单无商品/支付/评价子记录 | 不同分析的可用样本不同 | 记录覆盖，不默认填0或删除 |
| review_id重复、每单可多评价 | 直接连接会重复订单 | 检查来源，未来确定评价聚合方式 |
| 地理前缀多行且有整行重复 | 连接膨胀，不能当唯一维表 | 标记；有地理用途时再定映射 |
| 品类缺失、翻译缺失 | 品类画像覆盖不全 | 区分两类缺失，保留来源信息 |
| 时间字段缺失及边界稀疏 | 事件定义、观察期受限制 | 先审查，不凭最大日期判完整 |
| 数值极端值 | 可能是真实大订单或错误 | 用分布与原始记录核查，不机械删尾部 |

统计字段之间的相关性属于后续EDA。先明确哪些是编号、类别、时间、数量，再选适合的比较；不要把ID编码后对52个字段盲算相关系数。

## 8. 本课交付：你应该能画出的结构与下一步
1. 给出九表各自粒度和52个字段的用途。
2. 解释主键/复合键，并指出reviews与geo不能按名字假定主键。
3. 画出关系、写出连接键、说明1:1/1:N/多对多风险。
4. 解释2条明细×2条支付为何变4行，说明何时先汇总。
5. 记录时间范围与质量问题，但不把尚未确定的业务排除规则当成清洗标准。

做到这些，再进入业务定义：服务谁 → 支持什么决定 → 分析单位 → 预测时点 → 结果事件及期限。
如果选复购，再定义什么叫购买/再次购买；如果选配送或评价，就定义相应目标，不预设复购。
随后按预测时点检查可用信息和标签覆盖，设计训练/验证/测试，再拟合预处理与模型，避免让未来信息参与训练。

现在先复算第4节的真实订单连接。你不需要先认同任何购买资格或90天窗口，也能验证自己是否真的理解表关系。
